# Workshop 1.4: Rolling Windows and Shifting Data

Welcome to Workshop 1.4! Financial markets are dynamic, producing streams of prices that arrive sequentially over time. To make sense of price trends and avoid false trading signals, quantitative analysts rely on time-series transformations.

### Smoothing Noise and Aligning Timestamps

Raw daily prices fluctuate erratically due to short-term market noise. To reveal the underlying trend, we frequently compute **moving averages** over recent windows. In pandas, sliding window calculations are handled by the **`.rolling()`** method.

Furthermore, testing a strategy requires strict discipline regarding when information becomes known. If we use today's closing price to place a trade earlier this morning, our backtest cheats by looking into the future. We prevent this fatal mistake, known as **look-ahead bias**, using **`.shift()`**.

In this workshop, we will explore rolling statistical windows and master time shifting to build honest trading simulations.

> **Key Takeaway**: Rolling windows smooth market fluctuations, while `.shift()` aligns signals realistically to eliminate look-ahead bias.

## Topic 1: Creating a Time-Series DataFrame

Time-series data consists of observations recorded at regular intervals. In quantitative finance, these observations are typically daily closing prices or transaction volumes.

When working with dates in pandas, we always convert date strings into true `datetime` objects using `pd.to_datetime()`. This ensures pandas recognizes temporal ordering.

Let's construct a daily temperature dataset to build our time-series intuitions cleanly. Let's see:

In [1]:
import pandas as pd

# 5 days of daily observations:
data = {
  "Day": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
  "Temperature": [22, 24, 19, 21, 23]
}

df = pd.DataFrame(data)

# Convert Day column from text strings to true datetime objects:
df["Day"] = pd.to_datetime(df["Day"])

print(df)

         Day  Temperature
0 2024-01-01           22
1 2024-01-02           24
2 2024-01-03           19
3 2024-01-04           21
4 2024-01-05           23

Data types:
Day            datetime64[ns]
Temperature             int64
dtype: object


> **Key Takeaway**: Storing dates as proper `datetime` objects enables pandas to understand chronological sequence.

---

## Topic 2: .rolling(): Moving Windows

Think of a **rolling window** like a sliding magnifying glass moving down our table one row at a time. Specifying `window=3` instructs pandas to look through that glass and capture only the current row plus the two preceding rows.

Once the window captures those three points, we can compute any summary statistic, such as `.mean()`.

Let's calculate a 3-day moving average and inspect how the table updates. Let's see:

In [2]:
# Calculate a 3-day moving average:
df["Temperature_3day_avg"] = df["Temperature"].rolling(window=3).mean()

print(df)

         Day  Temperature  Temperature_3day_avg
0 2024-01-01           22                   NaN
1 2024-01-02           24                   NaN
2 2024-01-03           19             21.666667
3 2024-01-04           21             21.333333
4 2024-01-05           23             21.000000


### Understanding Why Initial Rows Show NaN

Notice that rows 0 and 1 display **`NaN`** (*Not a Number*, indicating a missing value). This is not an error or a software glitch!

To compute a 3-day average, pandas requires three complete historical observations:
- On Day 1, only one observation exists, so a 3-day average cannot be calculated.
- On Day 2, only two observations exist (`22` and `24`), which is still insufficient.
- On Day 3, three historical observations become available, yielding `(22 + 24 + 19) / 3 = 21.67`.

As we step forward to Day 4, the window rolls forward to include the newest temperature reading while dropping the oldest.

> **Key Takeaway**: A rolling window requires a full set of historical rows, naturally producing `NaN` values until the window size is satisfied.

## Topic 3: Rolling with Different Statistics

We are by no means restricted to moving averages! We can attach various statistical aggregations to a rolling window:
- **`.rolling(2).sum()`**: Calculates a cumulative 2-day sum, useful for monitoring multi-day trading volume.
- **`.rolling(3).std()`**: Computes a 3-day rolling standard deviation, providing a direct gauge of short-term volatility and risk.

Let's calculate both a rolling sum and rolling volatility. Let's see:

In [3]:
# We compute a 2-day rolling sum:
df["Temp_2day_sum"] = df["Temperature"].rolling(2).sum()

# We compute a 3-day rolling standard deviation to measure volatility:
df["Temp_3day_std"] = df["Temperature"].rolling(3).std()

print(df[["Day", "Temperature", "Temp_2day_sum", "Temp_3day_std"]])

         Day  Temperature  Temp_2day_sum  Temp_3day_std
0 2024-01-01           22            NaN            NaN
1 2024-01-02           24           46.0            NaN
2 2024-01-03           19           43.0       2.516611
3 2024-01-04           21           40.0       2.516611
4 2024-01-05           23           44.0       2.000000


> **Key Takeaway**: Rolling aggregations like `.sum()` and `.std()` allow us to track cumulative volume and evolving volatility over time.

---

## Topic 4: .shift(): Shifting Data to Create Lags

To answer market questions like *"Did today's price close higher than yesterday's?"*, we need yesterday's closing figure placed directly alongside today's record.

In pandas, we shift data points along the index using **`.shift()`**:
- **Positive Shift `.shift(1)`**: Pushes data down by one row, placing yesterday's value next to today's date.
- **Negative Shift `.shift(-1)`**: Pulls data up by one row, positioning tomorrow's value next to today's date.

Let's see both lagging and leading shifts in action. Let's check:

In [4]:
# Lag by 1 day (yesterday's value):
df["Yesterday_Temp"] = df["Temperature"].shift(1)

# Lead by 1 day (tomorrow's value):
df["Tomorrow_Temp"] = df["Temperature"].shift(-1)

print(df[["Day", "Temperature", "Yesterday_Temp", "Tomorrow_Temp"]])

         Day  Temperature  Yesterday_Temp  Tomorrow_Temp
0 2024-01-01           22             NaN           24.0
1 2024-01-02           24            22.0           19.0
2 2024-01-03           19            24.0           21.0
3 2024-01-04           21            19.0           23.0
4 2024-01-05           23            21.0            NaN


Notice how the values align: on `2024-01-02`, today's reading is `24`, while `Yesterday_Temp` shows `22.0` from January 1. On the very first row, `Yesterday_Temp` displays `NaN` because no prior history exists.

> **Key Takeaway**: `.shift(1)` shifts observations down by one row, making past data available on the current row.

## Topic 5: Avoiding Look-Ahead Bias with .shift()

In quantitative finance, the most devastating mistake beginners make is introducing **look-ahead bias**, which means unintentionally peeking into future data during a simulation.

Suppose our strategy rule specifies:
> *"If the 3-day average temperature exceeds 20, buy tomorrow."*

When do we know today's 3-day average? Only after today's market has closed. Therefore, we cannot execute a trade today based on today's closing number. In the real world, our order can only enter the market on the following morning at the market open.

To simulate real trading faithfully, we **must lag our signal by 1 period** using `.shift(1)`. This shifts our decision into tomorrow's trading slot.

Let's generate a raw signal and lag it into an actionable execution signal. Let's see:

In [5]:
# Raw signal: 1 if 3-day average > 20, else 0:
df["Signal"] = (df["Temperature_3day_avg"] > 20).astype(int)

# Lag the signal by 1 day to represent realistic execution on the NEXT day:
df["Trading_Signal"] = df["Signal"].shift(1)

print(df[["Day", "Temperature_3day_avg", "Signal", "Trading_Signal"]])

         Day  Temperature_3day_avg  Signal  Trading_Signal
0 2024-01-01                   NaN       0             NaN
1 2024-01-02                   NaN       0             0.0
2 2024-01-03             21.666667       1             0.0
3 2024-01-04             21.333333       1             1.0
4 2024-01-05             21.000000       1             1.0


Observe how the timing shifts: on `2024-01-03`, `Signal` evaluates to 1 at the end of the day. But our `Trading_Signal` does not become active until `2024-01-04`. This lag guarantees our backtest respects chronological reality.

> **Key Takeaway**: Always lag trading signals by one period with `.shift(1)` to ensure your simulation strictly executes after signals become known.

---

## Practice Time

Now it is your turn to calculate rolling windows and lag signals with `.shift()`. Ensuring time-series integrity is what separates realistic quant models from broken simulations, so work through these steps with care.

---

### Challenge 1: Calculating Moving Averages

- Construct a 7-day DataFrame from this dictionary:
  ```python
  data_7 = {
      "Day": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-06", "2024-01-07"],
      "Temperature": [20, 22, 21, 25, 24, 26, 23]
  }
  ```
- Convert `Day` to datetime format using `pd.to_datetime()`.
- Add a column named `"MA_2day"` calculating a 2-day rolling average using `.rolling(2).mean()`.
- Display the DataFrame.

In [ ]:
# Challenge 1: Write your code below this line


# Expected Output:
#     Day Temperature MA_2day
# 0 2024-01-01      20   NaN
# 1 2024-01-02      22   21.0
# 2 2024-01-03      21   21.5
# 3 2024-01-04      25   23.0
# 4 2024-01-05      24   24.5
# 5 2024-01-06      26   25.0
# 6 2024-01-07      23   24.5


### Challenge 2: Generating Price Lags

- Using your 7-day DataFrame from Challenge 1:
- Add a column named `"Prev_Day_Temp"` containing the prior day's temperature using `.shift(1)`.
- Display the updated table to confirm the alignment.

In [ ]:
# Challenge 2: Write your code below this line


# Expected Output:
#     Day Temperature MA_2day Prev_Day_Temp
# 0 2024-01-01      20   NaN      NaN
# 1 2024-01-02      22   21.0      20.0
# 2 2024-01-03      21   21.5      22.0
# 3 2024-01-04      25   23.0      21.0
# 4 2024-01-05      24   24.5      25.0
# 5 2024-01-06      26   25.0      24.0
# 6 2024-01-07      23   24.5      26.0


### Challenge 3: Aligning Execution Signals

- Create a column `"Signal"`: set to 1 if `Temperature > Prev_Day_Temp`, else 0.
- Lag that signal by 1 period into a new column named `"Execution_Signal"` using `.shift(1)`.
- Display columns `["Day", "Temperature", "Prev_Day_Temp", "Signal", "Execution_Signal"]`.

In [ ]:
# Challenge 3: Write your code below this line


# Expected Output:
#     Day Temperature Prev_Day_Temp Signal Execution_Signal
# 0 2024-01-01      20      NaN    0        NaN
# 1 2024-01-02      22      20.0    1        0.0
# 2 2024-01-03      21      22.0    0        1.0
# 3 2024-01-04      25      21.0    1        0.0
# 4 2024-01-05      24      25.0    0        1.0
# 5 2024-01-06      26      24.0    1        0.0
# 6 2024-01-07      23      26.0    0        1.0


---

## Solutions Section

Outstanding work completing these time-series exercises! Knowing when to smooth data and how to prevent look-ahead bias is fundamental to rigorous quantitative testing.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
data_7 = {
    "Day": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-06", "2024-01-07"],
    "Temperature": [20, 22, 21, 25, 24, 26, 23]
}
df7 = pd.DataFrame(data_7)
df7["Day"] = pd.to_datetime(df7["Day"])
df7["MA_2day"] = df7["Temperature"].rolling(2).mean()
print(df7)
```

#### Solution for Challenge 2:
```python
df7["Prev_Day_Temp"] = df7["Temperature"].shift(1)
print(df7)
```

#### Solution for Challenge 3:
```python
df7["Signal"] = (df7["Temperature"] > df7["Prev_Day_Temp"]).astype(int)
df7["Execution_Signal"] = df7["Signal"].shift(1)
print(df7[["Day", "Temperature", "Prev_Day_Temp", "Signal", "Execution_Signal"]])
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [6]:
# Solution for Challenge 1:
data_7 = {
  "Day": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-06", "2024-01-07"],
  "Temperature": [20, 22, 21, 25, 24, 26, 23]
}
df7 = pd.DataFrame(data_7)
df7["Day"] = pd.to_datetime(df7["Day"])
df7["MA_2day"] = df7["Temperature"].rolling(2).mean()
print(df7)

         Day  Temperature  MA_2day
0 2024-01-01           20      NaN
1 2024-01-02           22     21.0
2 2024-01-03           21     21.5
3 2024-01-04           25     23.0
4 2024-01-05           24     24.5
5 2024-01-06           26     25.0
6 2024-01-07           23     24.5


In [7]:
# Solution for Challenge 2:
df7["Prev_Day_Temp"] = df7["Temperature"].shift(1)
print(df7)

         Day  Temperature  MA_2day  Prev_Day_Temp
0 2024-01-01           20      NaN            NaN
1 2024-01-02           22     21.0           20.0
2 2024-01-03           21     21.5           22.0
3 2024-01-04           25     23.0           21.0
4 2024-01-05           24     24.5           25.0
5 2024-01-06           26     25.0           24.0
6 2024-01-07           23     24.5           26.0


In [8]:
# Solution for Challenge 3:
df7["Signal"] = (df7["Temperature"] > df7["Prev_Day_Temp"]).astype(int)
df7["Execution_Signal"] = df7["Signal"].shift(1)
print(df7[["Day", "Temperature", "Prev_Day_Temp", "Signal", "Execution_Signal"]])

         Day  Temperature  Prev_Day_Temp  Signal  Execution_Signal
0 2024-01-01           20            NaN       0               NaN
1 2024-01-02           22           20.0       1               0.0
2 2024-01-03           21           22.0       0               1.0
3 2024-01-04           25           21.0       1               0.0
4 2024-01-05           24           25.0       0               1.0
5 2024-01-06           26           24.0       1               0.0
6 2024-01-07           23           26.0       0               1.0
